# 📊 AI Business Assistant — Complete ML & NLP Pipeline (`main.ipynb`)

This notebook documents the complete end-to-end workflow and execution sequence of the **AI Business Assistant** project:
1. **Data Preparation & Synthesis (`backend/data_prep.py`)** — Generating/cleaning store transaction records and bilingual customer reviews without requiring external CSV downloads.
2. **Exploratory Data Analysis & Business KPIs (`backend/analytics.py`)** — Calculating total revenue, orders, unique customers, Average Order Value (AOV), top/bottom products, and monthly sales trends using **Pandas**.
3. **Time-Series Sales Forecasting (`backend/forecasting.py`)** — Feature engineering (lag features, rolling statistics, calendar features) and training a **Random Forest Regressor** vs. a baseline **Dummy Regressor** to forecast the next 30 days of sales.
4. **Customer Review Sentiment Analysis (`backend/sentiment.py`)** — Training an NLP pipeline using **TF-IDF Vectorization** and **Logistic Regression** to classify customer reviews (`positive`, `neutral`, `negative`) and extract top customer complaints.
5. **Smart Business Chatbot & Actionable Recommendations (`backend/assistant.py`)** — Answering business owners' questions in **English and Arabic** based on real data and generating strategic recommendations.
6. **Saving Trained Models (`models/`)** — Exporting the trained models and evaluation metrics for the **FastAPI Backend** and **Streamlit Frontend**.

In [1]:
import os
import sys
import json
import joblib
import pandas as pd
import numpy as np

# Add project root to path so we can import the backend package
sys.path.insert(0, os.path.abspath('..'))

from backend.data_prep import load_sales, load_reviews
from backend import analytics as A
from backend.forecasting import train_forecaster, forecast_future, feature_importance_df
from backend.sentiment import train_sentiment, predict_sentiment, top_complaints
from backend.assistant import answer, recommendations

sales = load_sales()
reviews = load_reviews()
print(f'Sales dataset shape: {sales.shape} | Reviews dataset shape: {reviews.shape}')

Sales dataset shape: (12545, 10) | Reviews dataset shape: (1600, 2)


## Step 1: Inspect Cleaned Sales & Customer Reviews Data
We standardize columns, remove cancelled invoices (starting with `C`), filter positive quantities and unit prices, and compute `Revenue = Quantity * UnitPrice`.

In [2]:
print('--- Sample Sales Transactions ---')
print(sales.head(5).to_string(index=False))
print('\n--- Sample Customer Reviews ---')
print(reviews.head(5).to_string(index=False))

--- Sample Sales Transactions ---
InvoiceNo StockCode                               Description  Quantity         InvoiceDate  UnitPrice  CustomerID              Country  Revenue      Month
   500001   SKU-101       Wireless Noise-Canceling Headphones         2 2025-10-01 14:00:00      89.99        1294       United Kingdom   179.98 2025-10-01
   500001   SKU-112         Braided HDMI 2.1 High-Speed Cable         2 2025-10-01 15:00:00      14.50        1294       United Kingdom    29.00 2025-10-01
   500002   SKU-110 Smart LED Desk Lamp with Wireless Charger         2 2025-10-01 14:00:00      45.00        1058         Saudi Arabia    90.00 2025-10-01
   500003   SKU-106         Portable 20000mAh Fast Power Bank         1 2025-10-01 20:00:00      49.99        1370         Saudi Arabia    49.99 2025-10-01
   500004   SKU-104         Ergonomic Vertical Wireless Mouse         3 2025-10-01 17:00:00      34.99        1372 United Arab Emirates   104.97 2025-10-01

--- Sample Customer Reviews -

## Step 2: Exploratory Data Analysis (EDA) & Core KPIs
Using **Pandas**, we aggregate overall store KPIs, rank the top-performing and lowest-performing products, and examine monthly revenue progression.

In [3]:
store_kpis = A.kpis(sales)
print('Store KPIs:', json.dumps(store_kpis, indent=2))

print('\n--- Top 5 Best-Selling Products ---')
print(A.top_products(sales, n=5, ascending=False).to_string(index=False))

print('\n--- Recent 6 Months Revenue Trend ---')
print(A.monthly_sales(sales).tail(6).to_string(index=False))

Store KPIs: {
  "revenue": 1270826.48,
  "orders": 7887,
  "customers": 449,
  "avg_order": 161.13,
  "units_sold": 21937
}

--- Top 5 Best-Selling Products ---
                        Description   Revenue  Quantity  Orders
            Smart Fitness Watch Pro 314167.00      2426    1381
Wireless Noise-Canceling Headphones 240363.29      2671    1548
     Mechanical RGB Gaming Keyboard 134976.00      2109    1220
  Portable 20000mAh Fast Power Bank 118926.21      2379    1365
          USB-C 8-in-1 Aluminum Hub  93072.00      2216    1274

--- Recent 6 Months Revenue Trend ---
     Month   Revenue  Orders  Units
2026-04-01 107692.18     666   1847
2026-05-01 111011.76     703   1915
2026-06-01 109389.45     692   1880
2026-07-01 123822.30     766   2174
2026-08-01 129246.66     790   2254
2026-09-01 120414.37     762   2093


## Step 3: Sales Forecasting with Random Forest (`RandomForestRegressor`)
We aggregate daily revenue and engineer time-series features:
- **Lag features**: `lag_1`, `lag_2`, `lag_3`, `lag_7`, `lag_14`
- **Rolling window statistics**: 7-day mean (`roll_7_mean`), 14-day mean (`roll_14_mean`), and 7-day standard deviation (`roll_7_std`)
- **Calendar indicators**: Day of week (`dow`), weekend flag (`is_weekend`), `month`, and `day`

We evaluate on a strict chronological split (last 60 days as test set) and compare against `DummyRegressor` before forecasting the next 30 days.

In [4]:
rf_model, rf_metrics, rf_comp, feat_df = train_forecaster(sales, test_days=60, save_model=True)
future_30 = forecast_future(rf_model, sales, days=30)

print('--- Forecasting Evaluation (Test = Last 60 Days) ---')
for model_name, m in rf_metrics.items():
    print(f"{model_name:13s} -> MAE: ${m['MAE']:,.2f} | RMSE: ${m['RMSE']:,.2f} | R2: {m['R2']}")

print('\n--- Next 30 Days Sales Forecast Summary ---')
print(f"Projected 30-Day Total Revenue: ${future_30['Forecast'].sum():,.2f}")
print(f"Projected Daily Average:        ${future_30['Forecast'].mean():,.2f}")

--- Forecasting Evaluation (Test = Last 60 Days) ---
RandomForest  -> MAE: $750.81 | RMSE: $924.42 | R2: 0.446
DummyBaseline -> MAE: $1,077.07 | RMSE: $1,404.35 | R2: -0.278

--- Next 30 Days Sales Forecast Summary ---
Projected 30-Day Total Revenue: $104,719.65
Projected Daily Average:        $3,490.65


## Step 4: Customer Sentiment Analysis (`TF-IDF` + `LogisticRegression`)
We vectorize customer reviews using uni-gram and bi-gram TF-IDF (`ngram_range=(1, 2)`, `sublinear_tf=True`) and train a balanced multinomial **Logistic Regression** classifier. We also extract the most common negative bigrams to pinpoint customer complaints.

In [5]:
sent_model, sent_metrics = train_sentiment(reviews, save_model=True)
print(f"Sentiment Accuracy: {sent_metrics['accuracy']*100:.2f}% | Macro F1-Score: {sent_metrics['f1_macro']*100:.2f}%\n")

print('Top Customer Complaints (Negative Reviews):')
for phrase, count in top_complaints(reviews, n=5):
    print(f' - {phrase}: {count} mentions')

print('\nLive Sentiment Predictions:')
for sample_txt in ['Great build quality, and very fast delivery.', 'التوصيل متأخر جداً والجودة سيئة']:
    res = predict_sentiment(sent_model, sample_txt)
    print(f" '{sample_txt}' -> {res['sentiment'].upper()} ({res['confidence']}%)")

Sentiment Accuracy: 98.75% | Macro F1-Score: 98.75%

Top Customer Complaints (Negative Reviews):
 - stopped working: 84 mentions
 - working three: 84 mentions
 - three days: 84 mentions
 - price expensive: 74 mentions
 - expensive what: 74 mentions

Live Sentiment Predictions:
 [EN] 'Great build quality, and very fast delivery.' -> POSITIVE (100.0%)
 [AR] 'التوصيل متأخر جداً والجودة سيئة' -> NEGATIVE (99.9%)


## Step 5: Smart Business Chatbot & Actionable Recommendations
Every response from the assistant is grounded in the real KPIs, Random Forest forecast, and sentiment analysis results (supporting both English and Arabic).

In [6]:
print('Q (EN): What are the top selling products?')
print(answer('What are the top selling products?', sales, reviews, future_30, lang='en'))

print('\nQ (AR): هل المبيعات متوقع تزيد ولا تقل؟')
print(answer('هل المبيعات متوقع تزيد ولا تقل؟', sales, reviews, future_30, lang='ar'))

print('\n--- Actionable Business Recommendations ---')
for i, rec in enumerate(recommendations(sales, reviews, future_30, lang='en'), start=1):
    print(f'{i}. {rec}')

Q (EN): What are the top selling products?
**Top-Selling Products by Revenue:**
- **Smart Fitness Watch Pro**: $314,167.00 (2,426 units)
- **Wireless Noise-Canceling Headphones**: $240,363.29 (2,671 units)
- **Mechanical RGB Gaming Keyboard**: $134,976.00 (2,109 units)
- **Portable 20000mAh Fast Power Bank**: $118,926.21 (2,379 units)
- **USB-C 8-in-1 Aluminum Hub**: $93,072.00 (2,216 units)

Q (AR): هل المبيعات متوقع تزيد ولا تقل؟
**ملخص التنبؤ للمبيعات خلال 30 يوم:**
- **إجمالي الإيرادات المتوقعة:** $104,719.65
- **المتوسط اليومي المتوقع:** $3,490.65
- **الاتجاه المتوقع:** انخفاض 📉 (-17.7%)

--- Actionable Business Recommendations ---
1. **Protect Core Revenue Driver:** 'Smart Fitness Watch Pro' generates your highest revenue ($314,167) — maintain strict inventory replenishment to prevent stockouts.
2. **Counter Projected Slowdown:** Sales are forecasted to dip (-17.7%) — launch targeted promotional campaigns and discount bundles.
3. **Resolve Customer Pain Points:** Prioritize fixin

## Step 6: Save Models & Evaluation Metadata to `models/`
Finally, we save the trained models (`models/forecaster.joblib` and `models/sentiment.joblib`) and evaluation summary (`models/metrics.json`).

In [7]:
os.makedirs('../models', exist_ok=True)
joblib.dump(rf_model, '../models/forecaster.joblib')
joblib.dump(sent_model, '../models/sentiment.joblib')
print('Saved trained models and metrics to ../models/')

Saved trained models and metrics to ../models/:
 - forecaster.joblib
 - sentiment.joblib
 - metrics.json
